# Leveraging Visualizations and MLflow for In-depth Model Analysis

## Introduction

In any machine learning project, understanding the behavior, performance, and characteristics of the developed models is important. Clear, informative visualizations play a crucial role in this understanding, offering insights into the model's patterns, errors, and efficiency.

In this portion of the guide, we look at a notebook that is concerned with the generation and storage of common and useful plots associated with a regression task.

We'll be looking at two primary means of logging plots along with our logged models:

* <b>Direct plot logging</b> via mlflow.log_figure() we will use an in-memory figure reference to a generated plot.
* <b>Logging a local plot file</b> via mlflow.log_artifact() to allow us to log a locally stored image to the run.

## Role of Visualizations in Model Analysis

Visualizations act as a window into the intricate world of machine learning models. They enable the exploration of various aspects:

* <b>Understanding Data:</b> Initial visualizations allow for a deep dive into the data, revealing patterns, anomalies, and relationships that can inform the entire modeling process.
* <b>Model Evaluation:</b> Plots such as residual plots and prediction error plots help in diagnosing issues with the model and evaluating its performance.
* <b>Hyperparameter Tuning:</b> Visualization aids in understanding the impact of different hyperparameters on the model's performance, guiding the selection process.
* <b>Error Analysis:</b> They help in analyzing the types and patterns of errors made by the model, providing insight into possible improvements.



## A Warning about Procedural Generation of Plots
In the accompanying notebook to this subsection of this guide, we will observe plots being declared within functions. This approach deviates from the typical examples seen in machine learning tutorials and guides, so it's essential to clarify why this method is chosen for the provided examples.

<b> The Central Issue: Statefulness </b>

![MLflow Architecture](../../../../images/Logging_a_plot.png)




Notebooks inherently maintain a state across cells. While this feature can be beneficial, it poses significant challenges for ensuring the reliability and accuracy of code and output, especially for visualizations.

<b>The Challenges of Out-of-Order Execution</b>

One of the most significant issues in notebook environments is the potential for out-of-order execution. Cells can be run in any sequence, leading to a state where variables or outputs do not reflect the latest code changes. This problem is particularly acute for visualizations. If a plot is generated and then displayed in a separate cell, running the cells out of order can lead to displaying outdated or incorrect visualizations.

<b>Ensuring Accurate Visualizations</b>

For visualizations to serve their purpose of conveying accurate, clear, and reliable information, they must correspond to the current state of the data and model. Ensuring this correspondence in a notebook environment requires careful management of cell execution order and state, which can be cumbersome and error-prone.

<b>Why Use Functions to Generate Plots</b>

To mitigate these challenges, the example code opts for declaring plots within functions. This approach offers several advantages:

* <b>Encapsulation:</b> By encapsulating the plot generation within a function, the code ensures that the plot is generated with the current state of the data every time the function is called. This encapsulation avoids the pitfalls of out-of-order cell execution affecting the plot's accuracy.
* <b>Flexibility and Reusability:</b> Functions provide the flexibility to generate plots with different parameters and data without duplicating code. This reusability enhances code maintainability and readability.
* <b>Integration with MLflow:</b> Functions seamlessly integrate with MLflow, allowing for plots to be logged alongside metrics, parameters, and models, ensuring that the visualizations correspond to the specific run and model state. This integration provides a reliable and consolidated view of the model, metrics, and plots in the MLflow UI, avoiding the disjointed view that can occur in notebooks.
* <b>Avoiding Display in Stdout:</b> The function-based approach avoids direct printing of plots to the notebook's stdout. Direct printing can clutter the notebook, increase the saved notebook's size, and lead to confusion with multiple plots displayed in the notebook. By logging plots directly in MLflow, the example code keeps the notebook clean, ensures plots correspond to the specific model run, and leverages MLflow's UI for viewing and comparing plots.
By encapsulating and scoping the generation of plots to within the training context (within <b>mlflow.start_run()</b>), we can get all of the flexibility, ease-of-use, and benefits of imperative iterative code development that notebooks bring without the risk of logging stale, invalid, or inaccurate plots that do not reflect the actual state of the data or model which is logged.



## Benefits of Integrating Visualizations with MLflow
Integrating visualizations with MLflow presents several substantial benefits:

* <b>Persistent Storage:</b> Storing visualizations alongside the model in MLflow ensures their availability for future reference, protecting against loss due to session termination or other issues.
* <b>Provenance:</b> It provides clear provenance for visualizations, ensuring that the insights they provide can always be traced back to the exact model version and dataset.
* <b>Consistency:</b> Ensures that the visualizations correspond to the correct version of the model, preventing confusion and errors.
* <b>Accessibility:</b> Makes visualizations easily accessible to all team members, enhancing collaboration and insight sharing.



## Generating a Plot
In the companion notebook to this section of the guide, there are samples of many regression-relevant plots. Some, such as the correlation matrix plot, are relevant to the feature data set, while others, such as the coefficients plot, are relevant only after we have a trained model.

Whether we're using a trained model or not, the approach is similar for logging these image artifacts.


### Defining a Plot

In the intricate world of data visualization, the structured and organized presentation of plots is paramount. Below is an example of generating a box plot, which compares a continuous variable to a categorical (ordinal) variable. The example utilizes a typical matplotlib implementation, enhanced with seaborn for a refined visual appearance. This structure is fundamental for ensuring clarity and legibility in our modeling code. By defining the plot generation as a separate, callable function, we maintain a clean and organized codebase. This approach is essential, especially in a notebook environment, to ensure that each training iteration has a specific and unambiguous reference to the plot generation, directly linked to the exact state of the data used in a training iteration. Such a method mitigates the risks associated with declaratively defined and materialized plots, which, if not regenerated after data modification, can lead to inconsistencies and errors in data representation.

In [1]:
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import math
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    median_absolute_error,
    r2_score,
)

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

In [2]:
def plot_box_weekend(df, style="seaborn-v0_8", plot_size=(10, 8)):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)
        sns.boxplot(data=df, x="weekend", y="demand", ax=ax, color="lightgray")
        sns.stripplot(
            data=df,
            x="weekend",
            y="demand",
            ax=ax,
            hue="weekend",
            palette={0: "blue", 1: "green"},
            alpha=0.15,
            jitter=0.3,
            size=5,
        )

        ax.set_title("Box Plot of Demand on Weekends vs. Weekdays", fontsize=14)
        ax.set_xlabel("Weekend (0: No, 1: Yes)", fontsize=12)
        ax.set_ylabel("Demand", fontsize=12)
        for i in ax.get_xticklabels() + ax.get_yticklabels():
            i.set_fontsize(10)
        ax.legend_.remove()
        plt.tight_layout()
    plt.close(fig)
    return fig

### Key Elements
* <b>Title Application:</b> Including a title in the plot is not just a formality, it's a necessity for ensuring clarity and comprehensibility, especially within the MLflow UI. A well-crafted title provides a comprehensive overview, helping in immediate understanding and eliminating any ambiguity or confusion.
* <b>Override Default Sizing:</b> Adjusting default sizes for various elements like fonts and plot sizes is crucial for ensuring the legibility and visual appeal of the plot in the MLflow UI. It ensures that the plot remains readable and clear, irrespective of the viewing platform or screen size.
* <b>Axes Labeling:</b> Properly labeled axes are a pillar of understandable and self-sufficient plots. They offer clear information about the data dimensions, making the plot comprehensible without external references or explanations.
* <b>Figure Closure:</b> Closing the figure before returning it ensures a clean and uncluttered notebook environment. It prevents the inadvertent display of the plot within the notebook's standard output, avoiding confusion and maintaining the organization of the notebook.
* <b>Legend Removal:</b> Removing auto-generated legends from the plot enhances the visual clarity and readability. It prevents unnecessary clutter, making the plot more concise and to the point, ensuring that the focus remains on the vital data representations.

## Defining a Plot to be Saved Locally
There are scenarios when saving a plot locally before logging to MLflow is more advantageous. The example below illustrates the generation of a correlation matrix plot, saving the image when called, as opposed to returning an in-memory reference. This approach, though different, remains seamlessly compatible with MLflow, ensuring the same level of organization and access, with additional flexibility in plot access and usage.

In [3]:
import tempfile

def plot_correlation_matrix_and_save(
    df,
    style="seaborn-v0_8",
    plot_size=(10, 8),
    path=str(pathlib.Path(tempfile.gettempdir()) / "corr_plot.png"),
):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)

        # Select only numeric columns so corr() works reliably
        numeric_df = df.select_dtypes(include=[np.number])
        corr = numeric_df.corr()

        mask = np.triu(np.ones_like(corr, dtype=bool))

        sns.heatmap(
            corr,
            mask=mask,
            cmap="coolwarm",
            vmax=0.3,
            center=0,
            square=True,
            linewidths=0.5,
            annot=True,
            fmt=".2f",
            ax=ax,
        )

        ax.set_title("Feature Correlation Matrix", fontsize=14)
        plt.tight_layout()

        # Save before closing the figure
        save_path = pathlib.Path(path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path)

    plt.close(fig)
    return str(save_path)

### Key Insights
* <b>Heatmap for Correlation:</b> The use of a heatmap in this context provides a visually intuitive and effective representation of feature correlations. It allows for easy identification of relationships between different features, enhancing the understandability and analysis depth.
* <b>Title and Layout Adjustments:</b> Including a clear and descriptive title, along with layout adjustments, ensures clarity and a compact presentation, enhancing the plot's usability and interpretation ease.
* <b>Local Saving of Plot:</b> Saving the figure locally provides easy access and reference, ensuring it's not tied to the notebook's execution state. It offers flexibility in access and ensures that the plot remains available independently, contributing to more organized and efficient data analysis and model evaluation processes.

### Logging plot images
In the below code snippet from the main notebook, we're executing our training and plot generations as a single atomic operation. As mentioned before, this helps to ensure that regardless of the state of any other cell within the notebook, the plots that are generated are going to refer to the state of the training data that was used to both train and evaluate the model.

For all of the plots apart from the correlation matrix, we're using the direct `matplotlib` `Figure` object reference for the plot when we call `mlflow.log_figure()`. For the correlation matrix, we're operating on a locally saved `.png` image file. This requires the usage of the more generic artifact writer (it supports any file type) `mlflow.log_artifact()`.

In [4]:
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)

# 1. Date range (e.g., 1 year of daily records)
dates = pd.date_range(start="2024-01-01", periods=365, freq="D")
n_samples = len(dates)

# 2. Base features
# weekend: 1 if Saturday (5) or Sunday (6), else 0
weekend = (dates.dayofweek >= 5).astype(int)

# price: continuous feature around $20 - $50
price = np.random.uniform(20.0, 50.0, size=n_samples)

# Additional numeric features for Ridge regression / correlation heatmap
promotion = np.random.binomial(n=1, p=0.2, size=n_samples)
temperature = np.random.normal(loc=22.0, scale=8.0, size=n_samples)

# 3. Target variable: demand
base_demand = 120
demand = (
    base_demand
    - 1.5 * price
    + 15.0 * weekend
    + 10.0 * promotion
    + 0.5 * temperature
    + np.random.normal(0, 5, size=n_samples)
)
demand = np.clip(demand, a_min=1.0, a_max=None)

# 4. Construct DataFrame
my_data = pd.DataFrame(
    {
        "date": dates,
        "weekend": weekend,
        "price": np.round(price, 2),
        "promotion": promotion,
        "temperature": np.round(temperature, 1),
        "demand": np.round(demand, 2),
    }
)

print(my_data.head())

        date  weekend  price  promotion  temperature  demand
0 2024-01-01        0  31.24          0         25.2   82.22
1 2024-01-02        0  48.52          1         18.6   72.51
2 2024-01-03        0  41.96          1         24.3   71.60
3 2024-01-04        0  37.96          0         38.6   79.57
4 2024-01-05        0  24.68          0         29.0   99.35


### Pre-Modeling Plots (fig1, fig3, fig4)

In [5]:
def plot_time_series_demand(
    df, window_size=28, style="seaborn-v0_8", plot_size=(10, 6)
):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)
        sorted_df = df.sort_values("date")
        ax.plot(
            sorted_df["date"],
            sorted_df["demand"],
            alpha=0.5,
            label="Actual Demand",
        )

        rolling_mean = (
            sorted_df["demand"].rolling(window=window_size, min_periods=1).mean()
        )
        ax.plot(
            sorted_df["date"],
            rolling_mean,
            color="red",
            linewidth=2,
            label=f"{window_size}-Day Rolling Mean",
        )

        ax.set_title("Time Series Demand", fontsize=14)
        ax.set_xlabel("Date", fontsize=12)
        ax.set_ylabel("Demand", fontsize=12)
        ax.legend()
        plt.tight_layout()
    plt.close(fig)
    return fig


def plot_scatter_demand_price(df, style="seaborn-v0_8", plot_size=(10, 6)):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)
        sns.scatterplot(
            data=df, x="price", y="demand", ax=ax, alpha=0.7, color="teal"
        )
        sns.regplot(
            data=df,
            x="price",
            y="demand",
            ax=ax,
            scatter=False,
            color="darkred",
        )

        ax.set_title("Demand vs. Price", fontsize=14)
        ax.set_xlabel("Price", fontsize=12)
        ax.set_ylabel("Demand", fontsize=12)
        plt.tight_layout()
    plt.close(fig)
    return fig


def plot_density_weekday_weekend(df, style="seaborn-v0_8", plot_size=(10, 6)):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)
        sns.kdeplot(
            data=df[df["weekend"] == 0]["demand"],
            label="Weekday",
            fill=True,
            ax=ax,
            alpha=0.4,
        )
        sns.kdeplot(
            data=df[df["weekend"] == 1]["demand"],
            label="Weekend",
            fill=True,
            ax=ax,
            alpha=0.4,
        )

        ax.set_title("Demand Density: Weekday vs. Weekend", fontsize=14)
        ax.set_xlabel("Demand", fontsize=12)
        ax.set_ylabel("Density", fontsize=12)
        ax.legend()
        plt.tight_layout()
    plt.close(fig)
    return fig

### Post-Modeling / Evaluation Plots (fig5, fig6, fig7, fig8)

In [6]:
def plot_residuals(y_true, y_pred, style="seaborn-v0_8", plot_size=(10, 6)):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)
        residuals = y_true - y_pred
        sns.scatterplot(
            x=y_pred, y=residuals, ax=ax, alpha=0.6, color="royalblue"
        )
        ax.axhline(0, color="red", linestyle="--", linewidth=1.5)

        ax.set_title("Residuals vs. Fitted Values", fontsize=14)
        ax.set_xlabel("Predicted Values", fontsize=12)
        ax.set_ylabel("Residuals", fontsize=12)
        plt.tight_layout()
    plt.close(fig)
    return fig


def plot_coefficients(
    model, feature_names, style="seaborn-v0_8", plot_size=(10, 6)
):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)
        coef_series = pd.Series(model.coef_, index=feature_names).sort_values()
        coef_series.plot(kind="barh", ax=ax, color="steelblue")

        ax.set_title("Model Coefficients", fontsize=14)
        ax.set_xlabel("Coefficient Value", fontsize=12)
        ax.set_ylabel("Features", fontsize=12)
        plt.tight_layout()
    plt.close(fig)
    return fig


def plot_prediction_error(
    y_true, y_pred, style="seaborn-v0_8", plot_size=(10, 6)
):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)
        sns.scatterplot(
            x=y_true, y=y_pred, ax=ax, alpha=0.6, color="darkslateblue"
        )

        # 45-degree reference line
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        ax.plot(
            [min_val, max_val],
            [min_val, max_val],
            "r--",
            lw=2,
            label="Ideal Fit",
        )

        ax.set_title("Prediction Error (Actual vs. Predicted)", fontsize=14)
        ax.set_xlabel("Actual Demand", fontsize=12)
        ax.set_ylabel("Predicted Demand", fontsize=12)
        ax.legend()
        plt.tight_layout()
    plt.close(fig)
    return fig


def plot_qq(y_true, y_pred, style="seaborn-v0_8", plot_size=(10, 6)):
    with plt.style.context(style=style):
        fig, ax = plt.subplots(figsize=plot_size)
        residuals = y_true - y_pred
        stats.probplot(residuals, dist="norm", plot=ax)

        ax.set_title("Normal Q-Q Plot of Residuals", fontsize=14)
        plt.tight_layout()
    plt.close(fig)
    return fig

In [7]:
import mlflow


mlflow.set_tracking_uri("http://127.0.0.1:5001")

mlflow.set_experiment("Visualizations Demo")

X = my_data.drop(columns=["demand", "date"])
y = my_data["demand"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

fig1 = plot_time_series_demand(my_data, window_size=28)
fig2 = plot_box_weekend(my_data)
fig3 = plot_scatter_demand_price(my_data)
fig4 = plot_density_weekday_weekend(my_data)

# Execute the correlation plot, saving the plot to a local temporary directory
# Execute the correlation plot and store the path where it was saved
corr_plot_path = plot_correlation_matrix_and_save(my_data)

# Define our Ridge model
model = Ridge(alpha=1.0)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate error metrics
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
msle = mean_squared_log_error(y_test, y_pred)
medae = median_absolute_error(y_test, y_pred)

# Generate prediction-dependent plots
fig5 = plot_residuals(y_test, y_pred)
fig6 = plot_coefficients(model, X_test.columns)
fig7 = plot_prediction_error(y_test, y_pred)
fig8 = plot_qq(y_test, y_pred)

# Start an MLflow run for logging metrics, parameters, the model, and our figures
with mlflow.start_run() as run:
    # Log the model
    mlflow.sklearn.log_model(sk_model=model, input_example=X_test, name="model")

    # Log the metrics
    mlflow.log_metrics({
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "msle": msle,
        "medae": medae,
    })

    # Log the hyperparameter
    mlflow.log_param("alpha", 1.0)

    # Log plots
    mlflow.log_figure(fig1, "time_series_demand.png")
    mlflow.log_figure(fig2, "box_weekend.png")
    mlflow.log_figure(fig3, "scatter_demand_price.png")
    mlflow.log_figure(fig4, "density_weekday_weekend.png")
    mlflow.log_figure(fig5, "residuals_plot.png")
    mlflow.log_figure(fig6, "coefficients_plot.png")
    mlflow.log_figure(fig7, "prediction_errors.png")
    mlflow.log_figure(fig8, "qq_plot.png")

   # Log the saved correlation matrix plot using the stored path
    mlflow.log_artifact(corr_plot_path)

c:\Users\apsit\anaconda3\envs\tamal_torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\apsit\anaconda3\envs\tamal_torch\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`

🏃 View run judicious-pug-843 at: http://127.0.0.1:5001/#/experiments/6/runs/62038b449ddd42c48dad4086384ecbe5
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/6
